In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import window
from utils.spark_structure_streaming_plus import *
from delta.tables import DeltaTable

import shutil
import functools
import uuid

CATALOG = 'data_ai_dbx_learning_workspace'
SCHEMA = "default"
VOLUME = "sample_volume"
BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/adhoc_checkpoints"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

tckp_display = register_tckp_display(display, BASE_PATH)


In [0]:
def rate_events(spark, rows_per_second=10, silent_users=(0,1,2,3,4),
                active_s=15, silent_s=15, stagger_s=5):
    """
    Each user in `silent_users` repeats a cycle of `active_s` seconds of normal
    events followed by `silent_s` seconds with no events at all.
    stagger_s shifts each silent user's cycle by (user_id * stagger_s) seconds
    so they don't all go quiet at the same moment.
    """
    cycle_s = active_s + silent_s

    df = (
        spark.readStream.format("rate")
        .option("rowsPerSecond", rows_per_second)
        .load()
    )
    keep_prob = 0.2 + 0.8 * f.abs(f.sin(f.col("value") / 20.0))
    df = df.where(f.rand() < keep_prob)

    events = df.select(
        f.col("timestamp").alias("event_time"),
        f.col("value").alias("event_id"),
        (f.col("value") % 10).alias("user_id"),
        f.round(f.rand() * 100, 2).alias("amount"),
        f.element_at(
            f.array(f.lit("click"), f.lit("view"), f.lit("purchase")),
            (f.col("value") % 3 + 1).cast("int"),
        ).alias("event_type"),
    )

    # Position of each event inside its user's active/silent cycle
    phase = (f.col("event_time").cast("long") + f.col("user_id") * stagger_s) % cycle_s
    in_silence = phase >= active_s

    return (events
        .where(~(f.col("user_id").isin(*silent_users) & in_silence))
        .withWatermark("event_time", "10 seconds"))

events = rate_events(spark)

In [0]:
dim_user = (
    spark.range(0, 10)
    .withColumnRenamed("id", "user_id")
    # simple formatted key: user_00001
    .withColumn("username", f.concat(f.lit("user_"), f.lpad(f.col("user_id").cast("string"), 5, "0")))
    .withColumn("email", f.concat(f.col("username"), f.lit("@example.com")))

    # cycle through a fixed list using modulo on id
    .withColumn("country", f.element_at(
        f.array(*[f.lit(c) for c in ["CA", "US", "UK", "DE", "FR"]]),
        (f.col("user_id") % 5 + 1).cast("int")))          # element_at is 1-indexed

    # pseudo-random but deterministic per id (stable across reruns/partitioning)
    .withColumn("tier", f.element_at(
        f.array(f.lit("free"), f.lit("pro"), f.lit("enterprise")),
        (f.pmod(f.xxhash64("user_id"), f.lit(3)) + 1).cast("int")))
    .withColumn("age", 18 + f.pmod(f.xxhash64("user_id", f.lit("age")), f.lit(60)))

    # derived date: signup date spread over the last 2 years
    .withColumn("signup_date", f.expr("date_sub(current_date(), cast(pmod(xxhash64(user_id, 'signup'), 730) as int))"))

    # boolean flag, ~20% true
    .withColumn("is_active", f.pmod(f.xxhash64("user_id", f.lit("active")), f.lit(10)) >= 2)
)

dim_user.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_user")

In [0]:
# StreamingQueryListener objects observe the entire Spark session,
# where all streams within the session trigger events as long as it's defined after
# the StreamingQueryListener object
my_listener = TableStreamingQueryListener(spark, "stream_lifecycle", "stream_progress")
spark.streams.addListener(my_listener)

def process_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        pass

    (batch_df.alias("b")
        .filter(f.col("user_id") == 0)
        .join(spark.table("dim_user").alias("d"), on="user_id", how="left")
        .select("b.*", "d.username", "d.tier")
        .write
        .mode("append")
        .saveAsTable("streaming_results"))
    
streaming_query = (events.writeStream
    .queryName("demo_streaming_query")
    .foreachBatch(process_batch)
    .option("checkpointLocation", "/Volumes/data_ai_dbx_learning_workspace/default/sample_volume/adhoc_checkpoints/tmp_0")
    .start())

In [0]:
# Log tables are seperated into 2:
#   1. Lifecycle Table - Tracks stream starts (WIP) and termination
#   2. Progress Table - Tracks micro batch performance and progress

spark.table('data_ai_dbx_learning_workspace.default.stream_lifecycle').display()
spark.table('data_ai_dbx_learning_workspace.default.stream_progress').display()

In [0]:
# Simple exploratory display() queries require a checkpoint
# the tckp_display wrapper manages the checkpoint creation
# giving the user the same behavior of display on a regular table
# but on a streaming table

# Will raise an error, forcing manual entry of some checkpoint which cannot be reused
# events.display()

events.tckp_display()

In [0]:
# Session Windows will only release a window grouping once it confirms the session has closed
# sometimes processing the windowed event once its closed is too late
sw_events = (events
    .groupBy(f.session_window("event_time", "10 seconds"), "user_id")
    .agg(
        f.sum("amount").alias("sum"),
        f.count("*").alias("count")))

def sw_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    out_df = (batch_df
        .withColumn("user_event_id", f.sha(f.concat(f.col("user_id").cast("string"), f.col("session_window.start").cast("string")))))
    
    # Create table on first batch
    if not spark.catalog.tableExists('sw_streaming_results'):
        (out_df.limit(0)
            .write
            .format("delta")
            .option("delta.enableChangeDataFeed", "true")
            .mode("ignore")
            .saveAsTable('sw_streaming_results'))

    (DeltaTable.forName(spark, "sw_streaming_results").alias("t")
        .merge(out_df.alias("s"), "t.user_event_id = s.user_event_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

sw_streaming_query = (sw_events.writeStream
    .foreachBatch(sw_batch)
    .option("checkpointLocation", "/Volumes/data_ai_dbx_learning_workspace/default/sample_volume/adhoc_checkpoints/sw_events")
    .start())

In [0]:
# Notice this table will only insert records when Spark can
# guarantee no additional records belong to a session window 
cdf_df = (spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1)
    .table("data_ai_dbx_learning_workspace.default.sw_streaming_results"))

cdf_df.display()

In [0]:
# Live Session Windows will release a window grouping as soon as its created,
# sending updates until its closed which processing and
# updating the windowed event in real-time
lsw_events = (events
    .live_session_window(
        groupBy="user_id",
        agg=(
            f.sum("amount").alias("sum"),
            f.count("*").alias("count")),
        timeColumn=f.col("event_time"),
        gapDuration="10 seconds"))

def lsw_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    out_df = (batch_df
        .withColumn("user_event_id", f.sha(f.concat(f.col("user_id").cast("string"), f.col("session_start").cast("string")))))
    
    # Create table on first batch
    if not spark.catalog.tableExists('lsw_streaming_results'):
        (out_df.limit(0)
            .write
            .format("delta")
            .option("delta.enableChangeDataFeed", "true")
            .mode("ignore")
            .saveAsTable('lsw_streaming_results'))

    (DeltaTable.forName(spark, "lsw_streaming_results").alias("t")
        .merge(out_df.alias("s"), "t.user_event_id = s.user_event_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

sw_streaming_query = (lsw_events.writeStream
    .foreachBatch(lsw_batch)
    .option("checkpointLocation", "/Volumes/data_ai_dbx_learning_workspace/default/sample_volume/adhoc_checkpoints/lsw_events")
    .start())

In [0]:
# Instead the Live Session Window will insert the event as soon as possible
# but also send the complete windowed event like the native counterpart
cdf_df = (spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1)
    .table("data_ai_dbx_learning_workspace.default.lsw_streaming_results"))

cdf_df.display()

In [0]:
spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.dim_user")
spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.streaming_results")
spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.sw_streaming_results")
spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.lsw_streaming_results")
spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.stream_lifecycle")
spark.sql("DROP TABLE IF EXISTS data_ai_dbx_learning_workspace.default.stream_progress")

tckp_display.cleanup()